# 02 - Análise estatística

## População e amostra

A população observada é o conjunto de registros de hotel por dia e hotel por semana fornecido nos arquivos.

Neste notebook, os dados disponíveis são tratados como o conjunto observado da população analisada, e não como uma amostra aleatória retirada de um mercado maior. As conclusões são válidas principalmente para os 14 hotéis, períodos e registros fornecidos.

## Preparação dos dados


In [1]:
import pandas as pd

df_diario = pd.read_json("../dados/daily_market.jsonl", lines=True)
df_semanal = pd.read_json("../dados/weekly_performance.jsonl", lines=True)
df_avaliacoes = pd.read_json("../dados/reviews.jsonl", lines=True)

df_diario["date"] = pd.to_datetime(df_diario["date"])
df_semanal["week_start"] = pd.to_datetime(df_semanal["week_start"])
df_avaliacoes["week_start"] = pd.to_datetime(df_avaliacoes["week_start"])

print(f"Registros diários: {len(df_diario):,}")
print(f"Registros semanais: {len(df_semanal):,}")
print(f"Avaliações: {len(df_avaliacoes):,}")
print(f"Hotéis: {df_diario['hotel_id'].nunique()}")
print(f"Período: {df_diario['date'].min():%Y-%m-%d} a {df_diario['date'].max():%Y-%m-%d}")

Registros diários: 10,192
Registros semanais: 1,456
Avaliações: 12,077
Hotéis: 14
Período: 2027-01-04 a 2028-12-31


## 1. Retrato do mercado

### Ocupação típica por categoria

A ocupação varia por dia e cada hotel aparece muitas vezes. Primeiro calculamos a média de cada hotel ao longo do período; depois comparamos os hotéis dentro de cada `tier`. Assim, cada hotel recebe o mesmo peso e não confundimos milhares de dias com milhares de hotéis.

A média usa todos os valores e pode ser puxada por hotéis com ocupação muito alta ou baixa. A mediana é mais resistente a esses extremos. A tabela mostra ambas e a ocupação de cada hotel: use a mediana quando poucos hotéis ou valores extremos afastarem a média do centro; se os valores forem equilibrados e próximos, a média também pode descrever bem o típico. Como há poucos hotéis por categoria, a decisão deve ser cautelosa.

Neste conjunto, média e mediana são muito próximas em todos os tiers (diferenças de até 0,3 ponto percentual). Assim, a média descreve bem a ocupação típica observada em `boutique`, `midscale` e `upscale`. Em `economy` e `luxury` há apenas dois hotéis por grupo; com dois valores, média e mediana coincidem, então não há base suficiente para detectar valores extremos ou escolher uma das medidas com maior confiança.

In [2]:
ocupacao_hotel = (
    df_diario.groupby(["hotel_id", "tier"], as_index=False)
    .agg(ocupacao_media=("occupancy", "mean"))
)

ocupacao_tier = (
    ocupacao_hotel.groupby("tier")["ocupacao_media"]
    .agg(media="mean", mediana="median", minimo="min", maximo="max", hoteis="count")
)
ocupacao_tier[["media", "mediana", "minimo", "maximo"]] *= 100
ocupacao_tier = ocupacao_tier.rename(columns={
    "media": "media_pct",
    "mediana": "mediana_pct",
    "minimo": "minimo_pct",
    "maximo": "maximo_pct",
})

print("Resumo por tier (ocupação em %):")
display(ocupacao_tier.round(1))

print("Ocupação média de cada hotel no período (em %):")
display(
    ocupacao_hotel.assign(ocupacao_pct=ocupacao_hotel["ocupacao_media"] * 100)
    .sort_values(["tier", "ocupacao_pct"])[["tier", "hotel_id", "ocupacao_pct"]]
    .round(1)
)

Resumo por tier (ocupação em %):


,media_pct,mediana_pct,minimo_pct,maximo_pct,hoteis
tier,,,,,
boutique,75.8,75.7,74.8,76.9,3
economy,61.9,61.9,60.4,63.5,2
luxury,72.3,72.3,71.9,72.8,2
midscale,68.3,68.6,67.6,68.7,3
upscale,73.0,73.3,70.7,74.9,4


Ocupação média de cada hotel no período (em %):


,tier,hotel_id,ocupacao_pct
10,boutique,H11,74.8
1,boutique,H02,75.7
6,boutique,H07,76.9
11,economy,H12,60.4
7,economy,H08,63.5
4,luxury,H05,71.9
2,luxury,H03,72.8
8,midscale,H09,67.6
12,midscale,H13,68.6
9,midscale,H10,68.7


### Variabilidade da diária (ADR)

Para medir a mudança da diária ao longo do tempo, calculamos o desvio padrão do ADR diário de cada hotel e depois tiramos a média desses desvios por `tier`. Essa ordem foca na variação temporal dentro de cada hotel, em vez de misturá-la com diferenças permanentes de preço entre hotéis. O desvio padrão está na mesma unidade do ADR e, portanto, é mais fácil de interpretar que a variância (que fica em unidades ao quadrado). O `tier` com maior desvio padrão médio é o mais variável em termos absolutos. Como o preço-base dos tiers é diferente, também mostramos o coeficiente de variação: desvio padrão dividido pelo ADR médio, em porcentagem, para comparar a oscilação proporcional.

O tier `luxury` apresenta a maior variação absoluta: desvio padrão médio de 65,74 na unidade do ADR. A variância média desse tier é 4.332,60 unidades ao quadrado. Se a comparação for proporcional ao preço médio, o coeficiente de variação é ligeiramente maior em `upscale` (18,01%, contra 17,90% em `luxury`). Portanto, a resposta depende do sentido de “mais variável”: em valor absoluto, `luxury`; em proporção ao seu preço médio, `upscale`.

In [3]:
adr_hotel = (
    df_diario.groupby(["hotel_id", "tier"])["adr"]
    .agg(
        adr_medio="mean",
        desvio_padrao=lambda valores: valores.std(ddof=0),
        variancia=lambda valores: valores.var(ddof=0),
    )
    .reset_index()
)
adr_hotel["coef_variacao_pct"] = adr_hotel["desvio_padrao"] / adr_hotel["adr_medio"] * 100

adr_tier = (
    adr_hotel.groupby("tier")
    .agg(
        adr_medio_dos_hoteis=("adr_medio", "mean"),
        desvio_padrao_medio=("desvio_padrao", "mean"),
        variancia_media=("variancia", "mean"),
        coef_variacao_medio_pct=("coef_variacao_pct", "mean"),
    )
    .sort_values("desvio_padrao_medio", ascending=False)
)
display(adr_tier.round(2))
print(f"Tier com maior desvio padrão médio: {adr_tier.index[0]}")

,adr_medio_dos_hoteis,desvio_padrao_medio,variancia_media,coef_variacao_medio_pct
tier,,,,
luxury,367.23,65.74,4332.60,17.90
upscale,223.51,40.27,1629.44,18.01
boutique,211.05,37.91,1475.28,17.96
midscale,151.76,26.92,726.00,17.73
economy,91.22,15.76,249.42,17.27


Tier com maior desvio padrão médio: luxury


### Frequência das notas

A frequência absoluta é a contagem de avaliações em cada nota; a relativa é essa contagem dividida pelo total; e a acumulada soma as frequências desde 1 estrela até a nota da linha. A última coluna acumulada permite ler diretamente a porcentagem de avaliações até 3 estrelas.

No conjunto analisado, 55,8% das avaliações têm 3 estrelas ou menos.

In [4]:
frequencia = (
    df_avaliacoes["rating"].value_counts()
    .reindex([1, 2, 3, 4, 5], fill_value=0)
    .sort_index()
    .rename_axis("estrelas")
    .to_frame("frequencia_absoluta")
)
frequencia["frequencia_relativa_pct"] = (
    frequencia["frequencia_absoluta"] / frequencia["frequencia_absoluta"].sum() * 100
)
frequencia["frequencia_acumulada"] = frequencia["frequencia_absoluta"].cumsum()
frequencia["frequencia_acumulada_pct"] = frequencia["frequencia_relativa_pct"].cumsum()
display(frequencia.round(2))

percentual_ate_3 = df_avaliacoes["rating"].le(3).mean() * 100
print(f"Avaliações com 3 estrelas ou menos: {percentual_ate_3:.1f}%")

,frequencia_absoluta,frequencia_relativa_pct,frequencia_acumulada,frequencia_acumulada_pct
estrelas,,,,
1,1087,9.00,1087,9.00
2,2439,20.20,3526,29.20
3,3212,26.60,6738,55.79
4,2302,19.06,9040,74.85
5,3037,25.15,12077,100.00


Avaliações com 3 estrelas ou menos: 55.8%


## 2. Sazonalidade e eventos

Para encontrar períodos de maior movimento, comparamos as médias das observações diárias por mês e por dia da semana. A ocupação é apresentada em porcentagem; o ADR mantém a unidade registrada no arquivo. Ordenar as tabelas da maior para a menor média facilita identificar os períodos de pico.

Neste conjunto, agosto tem as maiores médias mensais de ocupação (84,34%) e ADR (263,72). Sábado é o dia da semana com maiores médias de ocupação (79,78%) e ADR (241,32).

In [5]:
meses = {
    1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril",
    5: "Maio", 6: "Junho", 7: "Julho", 8: "Agosto",
    9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro",
}

por_mes = (
    df_diario.groupby("month")
    .agg(ocupacao_pct=("occupancy", "mean"), adr_medio=("adr", "mean"))
    .rename(index=meses)
)
por_mes["ocupacao_pct"] *= 100

print("Meses com maior ocupação média:")
display(por_mes.sort_values("ocupacao_pct", ascending=False).round(2))
print("Meses com maior ADR médio:")
display(por_mes.sort_values("adr_medio", ascending=False).round(2))

ordem_dias = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
por_dia = (
    df_diario.groupby("weekday")
    .agg(ocupacao_pct=("occupancy", "mean"), adr_medio=("adr", "mean"))
    .reindex(ordem_dias)
)
nomes_dias = {
    "Monday": "Segunda-feira", "Tuesday": "Terça-feira", "Wednesday": "Quarta-feira",
    "Thursday": "Quinta-feira", "Friday": "Sexta-feira", "Saturday": "Sábado", "Sunday": "Domingo",
}
por_dia = por_dia.rename(index=nomes_dias)
por_dia["ocupacao_pct"] *= 100

print("Dias da semana com maior ocupação média:")
display(por_dia.sort_values("ocupacao_pct", ascending=False).round(2))
print("Dias da semana com maior ADR médio:")
display(por_dia.sort_values("adr_medio", ascending=False).round(2))

Meses com maior ocupação média:


,ocupacao_pct,adr_medio
month,,
Agosto,84.34,263.72
Julho,81.57,249.39
Junho,78.53,232.97
Setembro,76.04,224.76
Maio,73.55,212.15
Outubro,70.85,202.15
Abril,70.42,200.68
Março,66.45,188.43
Novembro,63.12,177.49


Meses com maior ADR médio:


,ocupacao_pct,adr_medio
month,,
Agosto,84.34,263.72
Julho,81.57,249.39
Junho,78.53,232.97
Setembro,76.04,224.76
Maio,73.55,212.15
Outubro,70.85,202.15
Abril,70.42,200.68
Março,66.45,188.43
Dezembro,62.04,180.60


Dias da semana com maior ocupação média:


,ocupacao_pct,adr_medio
weekday,,
Sábado,79.78,241.32
Sexta-feira,75.61,226.42
Quarta-feira,72.32,209.69
Quinta-feira,71.75,207.35
Terça-feira,70.34,203.45
Domingo,63.42,175.14
Segunda-feira,63.34,186.32


Dias da semana com maior ADR médio:


,ocupacao_pct,adr_medio
weekday,,
Sábado,79.78,241.32
Sexta-feira,75.61,226.42
Quarta-feira,72.32,209.69
Quinta-feira,71.75,207.35
Terça-feira,70.34,203.45
Segunda-feira,63.34,186.32
Domingo,63.42,175.14


### Noites com evento e noites comuns

Consideramos noite com evento quando `demand_event` não está vazio. Comparamos a média das noites com evento com a das noites comuns. A diferença de ocupação é expressa em pontos percentuais (por exemplo, de 60% para 65% são +5 pontos); para ADR mostramos a diferença na unidade original. A tabela geral responde pelo mercado, e a tabela por `tier` verifica se a diferença é semelhante entre categorias.

No mercado, as noites com evento têm em média +9,87 pontos percentuais de ocupação e +41,85 na unidade do ADR. O efeito não é igual entre tiers: a mudança de ADR vai de +18,20 em `economy` a +74,72 em `luxury`; a mudança de ocupação fica entre +9,22 e +10,15 pontos.

In [6]:
df_eventos = df_diario.assign(
    tem_evento=df_diario["demand_event"].fillna("").str.strip().ne("")
)

def diferenca_evento(grupo):
    medias = grupo.groupby("tem_evento")[["occupancy", "adr"]].mean()
    return pd.Series({
        "ocupacao_noites_comuns_pct": medias.loc[False, "occupancy"] * 100,
        "ocupacao_noites_evento_pct": medias.loc[True, "occupancy"] * 100,
        "mudanca_ocupacao_pontos": (medias.loc[True, "occupancy"] - medias.loc[False, "occupancy"]) * 100,
        "adr_noites_comuns": medias.loc[False, "adr"],
        "adr_noites_evento": medias.loc[True, "adr"],
        "mudanca_adr": medias.loc[True, "adr"] - medias.loc[False, "adr"],
    })

print("Diferença média no mercado:")
display(diferenca_evento(df_eventos).round(2).to_frame("valor").T)

print("Diferença por tier:")
efeito_por_tier = df_eventos.groupby("tier").apply(diferenca_evento)
display(efeito_por_tier.round(2))

Diferença média no mercado:


,ocupacao_noites_comuns_pct,ocupacao_noites_evento_pct,mudanca_ocupacao_pontos,adr_noites_comuns,adr_noites_evento,mudanca_adr
valor,70.3,80.17,9.87,204.4,246.24,41.85


Diferença por tier:


,ocupacao_noites_comuns_pct,ocupacao_noites_evento_pct,mudanca_ocupacao_pontos,adr_noites_comuns,adr_noites_evento,mudanca_adr
tier,,,,,,
boutique,75.18,84.83,9.65,208.36,249.97,41.61
economy,61.35,70.57,9.22,90.04,108.24,18.20
luxury,71.68,81.82,10.15,362.41,437.13,74.72
midscale,67.67,77.80,10.13,149.75,180.88,31.13
upscale,72.40,82.43,10.03,220.58,266.03,45.45


## 3. Relações entre variáveis

A correlação de Pearson resume a direção e a intensidade de uma relação linear: valores positivos indicam que as duas variáveis tendem a subir juntas; valores negativos, que uma tende a subir quando a outra cai; valores próximos de zero indicam pouca relação linear. Correlação não prova causa. Um terceiro fator — como temporada, evento, tamanho ou posicionamento do hotel — pode influenciar as duas variáveis e criar uma associação aparente (correlação espúria).

A relação entre ocupação e ADR é positiva no mercado (0,603) e também em cada tier (de 0,722 a 0,973). A intensidade muda após separar as categorias, então a análise agregada não conta toda a história; temporada e eventos também podem influenciar as duas medidas.

In [7]:
correlacao_mercado = df_diario["occupancy"].corr(df_diario["adr"])
correlacao_por_tier = pd.Series({
    tier: grupo["occupancy"].corr(grupo["adr"])
    for tier, grupo in df_diario.groupby("tier")
}, name="correlacao_pearson_ocupacao_adr")

print(f"Correlação no mercado inteiro: {correlacao_mercado:.3f}")
print("Correlação dentro de cada tier:")
display(correlacao_por_tier.to_frame().round(3))

Correlação no mercado inteiro: 0.603
Correlação dentro de cada tier:


,correlacao_pearson_ocupacao_adr
boutique,0.722
economy,0.973
luxury,0.952
midscale,0.961
upscale,0.954


### Nota média semanal e desempenho financeiro

Usamos `mean_rating` e `trevpar` da tabela semanal. O `trevpar` representa a receita total por quarto disponível, permitindo comparar hotéis de tamanhos diferentes melhor do que a receita total. Mantemos apenas semanas com as duas medidas registradas e calculamos a correlação geral e por `tier`.

Mesmo que a correlação seja alta, não podemos concluir que a nota causou o desempenho: temporada, eventos, preços e características do hotel podem afetar ambos. Além disso, uma nota semanal pode se basear em poucas avaliações; a quantidade de avaliações deve ser considerada ao interpretar o resultado.

A correlação geral é positiva, mas fraca (0,229). Dentro de cada tier, os valores ficam próximos de zero (de -0,060 a 0,040). Essa diferença sugere que a relação geral pode vir das diferenças entre categorias, e não de uma relação consistente entre nota e desempenho dentro de cada categoria; portanto, não é evidência de causalidade.

In [8]:
semanas_validas = df_semanal.dropna(subset=["mean_rating", "trevpar"])
correlacao_nota_receita = semanas_validas["mean_rating"].corr(semanas_validas["trevpar"])
correlacao_nota_por_tier = pd.Series({
    tier: grupo["mean_rating"].corr(grupo["trevpar"])
    for tier, grupo in semanas_validas.groupby("tier")
}, name="correlacao_pearson")

print(f"Semanas consideradas: {len(semanas_validas):,}")
print(f"Correlação entre nota média e TRevPAR: {correlacao_nota_receita:.3f}")
print("Correlação por tier:")
display(correlacao_nota_por_tier.to_frame().round(3))

Semanas consideradas: 1,429
Correlação entre nota média e TRevPAR: 0.229
Correlação por tier:


,correlacao_pearson
boutique,-0.019
economy,0.028
luxury,-0.016
midscale,-0.060
upscale,0.040


### Notas parciais e nota geral

Calculamos a correlação entre cada nota parcial (de 1 a 5) e a nota geral. As maiores correlações positivas indicam quais aspectos mais acompanham as notas gerais neste conjunto de avaliações. Isso mostra associação, não que mudar um aspecto isoladamente causará uma mudança específica na nota.

`cleanliness` tem a associação mais alta com a nota geral (0,901), seguida por conforto (0,885) e serviço (0,878). Localização apresenta uma associação bem menor (0,166).

In [9]:
notas_parciais = [
    "cleanliness", "comfort", "service_score", "location_score",
    "value_score", "sleep_quality",
]
correlacao_notas = (
    df_avaliacoes[["rating"] + notas_parciais]
    .corr()["rating"]
    .drop("rating")
    .sort_values(ascending=False)
    .rename("correlacao_com_nota_geral")
)
display(correlacao_notas.to_frame().round(3))

,correlacao_com_nota_geral
cleanliness,0.901
comfort,0.885
service_score,0.878
sleep_quality,0.867
value_score,0.809
location_score,0.166


## 4. Projeção: extrapolação para 2029

Escolhemos o hotel `H01` e a ocupação média anual, em vez da receita, para que o exemplo seja fácil de interpretar e não dependa do tamanho do hotel. Com dois anos consecutivos, a extrapolação linear simples mantém a mesma variação observada de 2027 para 2028: estimativa de 2029 = ocupação de 2028 + (ocupação de 2028 − ocupação de 2027).

Isso é extrapolação porque 2029 está fora do período observado. Interpolação seria estimar um valor entre pontos conhecidos. Antes de projetar, verificamos se há dias ausentes; para H01 não há lacunas no período observado, então não foi necessário interpolar. A projeção não deve ser tratada como previsão confiável: usa apenas dois pontos anuais, supõe tendência constante e não incorpora sazonalidade futura, eventos, mudanças de preço, capacidade ou mercado. O primeiro período vai de 4 de janeiro de 2027 a 31 de dezembro, portanto não cobre os três primeiros dias do ano. Como ocupação não pode ficar fora de 0% a 100%, qualquer resultado fora desse intervalo seria fisicamente impossível e indicaria que o modelo linear não serve para esse horizonte.

Para H01, a ocupação média observada é 73,63% em 2027 e 73,89% em 2028. Mantendo o aumento de 0,26 ponto percentual, a extrapolação simples resulta em 74,14% para 2029.

In [10]:
hotel_escolhido = "H01"
dados_hotel = df_diario[df_diario["hotel_id"].eq(hotel_escolhido)].copy()

datas_esperadas = pd.date_range(df_diario["date"].min(), df_diario["date"].max(), freq="D")
datas_por_hotel = dados_hotel["date"].drop_duplicates()
dias_ausentes = len(datas_esperadas.difference(datas_por_hotel))
print(f"Dias sem registro para {hotel_escolhido}: {dias_ausentes}")

ocupacao_anual = dados_hotel.groupby(dados_hotel["date"].dt.year)["occupancy"].mean().sort_index()
if len(ocupacao_anual) != 2 or ocupacao_anual.index[1] - ocupacao_anual.index[0] != 1:
    raise ValueError("A extrapolação requer exatamente dois anos consecutivos de dados.")

ano_1, ano_2 = ocupacao_anual.index
ocupacao_2029 = ocupacao_anual.iloc[1] + (ocupacao_anual.iloc[1] - ocupacao_anual.iloc[0])

projecao = pd.DataFrame({
    "ano": [ano_1, ano_2, ano_2 + 1],
    "ocupacao_pct": [ocupacao_anual.iloc[0] * 100, ocupacao_anual.iloc[1] * 100, ocupacao_2029 * 100],
    "tipo": ["observado", "observado", "extrapolação linear"],
}).set_index("ano")
display(projecao.round(2))

if not 0 <= ocupacao_2029 <= 1:
    print("Atenção: a extrapolação ficou fora do intervalo possível de ocupação (0% a 100%).")

Dias sem registro para H01: 0


,ocupacao_pct,tipo
ano,,
2027,73.63,observado
2028,73.89,observado
2029,74.14,extrapolação linear


## Limitações da análise

- **População analisada:** os dados representam os registros disponíveis dos hotéis observados. Não são uma amostra aleatória de todos os hotéis do mercado, portanto os resultados não podem ser generalizados automaticamente.
- **Comparação entre grupos:** alguns tiers têm poucos hotéis. A média e a mediana de grupos com dois ou três hotéis são pouco robustas diante de diferenças individuais.
- **Correlação:** uma correlação positiva ou negativa não demonstra causalidade. Temporada, eventos, preço, localização e características dos hotéis podem influenciar simultaneamente as variáveis e produzir relações espúrias.
- **Notas semanais:** uma média de nota pode representar poucas avaliações em uma semana. Isso torna o peso real de algumas observações potencialmente diferente da quantidade de registros.
- **Eventos:** a comparação mostra associação, mas não controla outros fatores que podem coincidir com os eventos.
- **Extrapolação:** com apenas dois pontos anuais, o método linear não captura sazonalidade, eventos futuros, mudanças de preço ou capacidade e alterações no mercado. A estimativa deve ser tratada como exemplo didático, não como previsão confiável.
- **Período parcial:** o ano de 2027 começa em 4 de janeiro nos registros. Por isso, a média anual observada desse ano não representa os três primeiros dias.

## Conclusão

### Retrato do mercado

Boutique apresentou a maior ocupação típica, 75,8%, seguida por upscale, 73,0%, e luxury, 72,3%. Economy apresentou a menor ocupação, 61,9%. As médias e medianas por tier são próximas, indicando que a média descreve bem o centro observado, especialmente nos grupos com três ou quatro hotéis. Luxury apresentou a maior variação absoluta do ADR, com desvio padrão médio de 65,74; quando a variabilidade é avaliada proporcionalmente ao preço, upscale ficou ligeiramente acima.

### Sazonalidade e eventos

Agosto apresentou a maior ocupação e o maior ADR médio mensal. Sábado apresentou as maiores médias de ocupação e ADR entre os dias da semana. No mercado, as noites com evento tiveram ocupação média 9,87 pontos percentuais maior e ADR 41,85 maior que as noites comuns. A mudança de ocupação foi semelhante entre os tiers, mas a mudança de ADR variou bastante.

### Relações entre variáveis

A ocupação e o ADR tiveram correlação positiva no mercado, 0,603, e correlações ainda mais fortes dentro dos tiers. A nota média semanal teve correlação geral positiva e fraca com o TRevPAR, 0,229, mas relações próximas de zero dentro de cada tier. Isso não permite concluir causalidade ou relação espúria apenas com estes resultados. Entre as notas parciais, limpeza apresentou a maior associação com a nota geral.

### Projeção

Mantendo a variação linear observada no hotel H01, a ocupação seria projetada em 74,14% para 2029, contra 73,63% em 2027 e 73,89% em 2028. A projeção depende de apenas dois anos, não incorpora os fatores do mercado e deve ser interpretada apenas como um exemplo de extrapolação.